In [2]:
# ============================================================
# 03e_rolling_transformer.ipynb
# Train Transformer for rolling 12h-ahead sepsis prediction
#
# Key differences from 03b:
#   - Output: single sigmoid neuron (binary classification)
#   - Loss: weighted BCE + focal loss
#   - Input: sliding 24h window at each hour (not admission prefix)
#   - Labels: binary per-hour (sepsis within next 12h?)
# ============================================================

import sys
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

PROJECT_ROOT = Path('C:/Users/20220505/Desktop/AI-sepsis')
sys.path.append(str(PROJECT_ROOT / 'src'))

OUTPUT_DIR = Path("C:/Users/20220505/Desktop/output path")

# ── Constants ─────────────────────────────────────────────────
LOOKBACK_W     = 24
HERO_HORIZON_H = 12
MIN_HOUR       = 4
MAX_HOURS      = 200
NUM_BINS       = 48

ALERT_THRESHOLDS = {
    'no_alert'  : (0.00, 0.10),
    'high_risk' : (0.10, 0.50),
    'critical'  : (0.50, 1.00),
}

def get_alert_tier(score):
    if score >= ALERT_THRESHOLDS['critical'][0]:
        return 'critical'
    elif score >= ALERT_THRESHOLDS['high_risk'][0]:
        return 'high_risk'
    else:
        return 'no_alert'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device : {device}")

# ── Load metadata ─────────────────────────────────────────────
print("\nLoading metadata...")

with open(str(OUTPUT_DIR / 'rich_feature_names.txt')) as f:
    rich_feature_names = f.read().splitlines()

with open(str(OUTPUT_DIR / 'feature_names.txt')) as f:
    feature_cols = f.read().splitlines()

stay_ids_order = pd.read_csv(
    str(OUTPUT_DIR / 'stay_ids_order.csv')
).squeeze().tolist()
stay_to_idx = {sid: i for i, sid in enumerate(stay_ids_order)}

print(f"  Rich features   : {len(rich_feature_names)}")
print(f"  Static features : {len(feature_cols)}")
print(f"  Stays           : {len(stay_ids_order):,}")

# ── Load tensors ──────────────────────────────────────────────
print("\nLoading X_rich_full (1.49 GB)...")
X_rich_full = np.load(str(OUTPUT_DIR / 'X_rich_full.npy'))
print(f"  Shape : {X_rich_full.shape}")

print("Loading static features...")
all_features  = pd.read_csv(str(OUTPUT_DIR / 'engineered_features.csv'))
static_lookup = (
    all_features
    .set_index('stay_id')[feature_cols]
    .fillna(0)
)
X_static_all = np.zeros(
    (len(stay_ids_order), len(feature_cols)),
    dtype=np.float32
)
for i, sid in enumerate(stay_ids_order):
    if sid in static_lookup.index:
        X_static_all[i] = static_lookup.loc[sid].values.astype('float32')
print(f"  Static shape : {X_static_all.shape}")

# ── Load rolling labels ───────────────────────────────────────
print("Loading rolling labels...")
label_df = pd.read_csv(str(OUTPUT_DIR / 'rolling_labels_12h.csv'))
print(f"  Shape : {label_df.shape}")

# ── Class weights from training set ──────────────────────────
train_labels = label_df[label_df['split'] == 'train']
n_pos = train_labels['label_12h'].sum()
n_neg = len(train_labels) - n_pos
pos_weight = n_neg / n_pos
print(f"\nClass weight  : pos_weight = {pos_weight:.1f}x")
print(f"  Positives   : {n_pos:,}")
print(f"  Negatives   : {n_neg:,}")
print("\nSetup complete ✓")

Device : cuda

Loading metadata...
  Rich features   : 25
  Static features : 127
  Stays           : 74,550

Loading X_rich_full (1.49 GB)...
  Shape : (74550, 200, 25)
Loading static features...
  Static shape : (74550, 127)
Loading rolling labels...
  Shape : (2296449, 4)

Class weight  : pos_weight = 61.5x
  Positives   : 25,527
  Negatives   : 1,570,516

Setup complete ✓


In [4]:
# ============================================================
# Cell 2: Rolling window dataset and Transformer architecture
# ============================================================

class RollingWindowDataset(Dataset):
    """
    Each sample is one (stay, hour) pair.
    Returns:
      x_seq    : (seq_len, 25) — last LOOKBACK_W hours of vitals
      x_static : (127,)       — static engineered features
      length   : int          — actual sequence length
      label    : float        — 1 if sepsis within next 12h
    """
    def __init__(self, label_df, X_rich_full,
                 X_static_all, stay_to_idx,
                 lookback=LOOKBACK_W):
        self.lookback   = lookback
        self.X_rich     = X_rich_full
        self.X_static   = X_static_all
        self.stay_to_idx= stay_to_idx
        self.samples    = []

        skipped = 0
        for _, row in label_df.iterrows():
            sid   = int(row['stay_id'])
            hour  = int(row['hour'])
            label = int(row['label_12h'])

            if sid not in stay_to_idx:
                skipped += 1
                continue

            self.samples.append((sid, hour, label))

        print(f"  Dataset built : {len(self.samples):,} samples "
              f"({skipped:,} skipped)")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sid, hour, label = self.samples[idx]
        stay_idx = self.stay_to_idx[sid]

        h_start  = max(0, hour - self.lookback)
        seq      = self.X_rich[stay_idx, h_start:hour, :]
        seq_len  = len(seq)

        x_seq    = torch.tensor(seq,   dtype=torch.float32)
        x_static = torch.tensor(
            self.X_static[stay_idx], dtype=torch.float32
        )
        return x_seq, x_static, seq_len, float(label)


def collate_rolling(batch):
    x_seqs, x_statics, lengths, labels = zip(*batch)
    max_len  = max(lengths)
    n        = len(x_seqs)
    n_feat   = x_seqs[0].shape[1]

    x_padded = torch.zeros(n, max_len, n_feat)
    for i, seq in enumerate(x_seqs):
        x_padded[i, :len(seq), :] = seq

    return (
        x_padded,
        torch.stack(x_statics),
        torch.tensor(lengths, dtype=torch.long),
        torch.tensor(labels,  dtype=torch.float32),
    )


# ── Rolling Transformer (binary output) ──────────────────────
class RollingTransformer(nn.Module):
    """
    Same architecture as TransformerSurvival but:
    - Output: single sigmoid score instead of softmax PMF
    - Designed for binary BCE training
    """
    def __init__(self,
                 vital_dim    = 25,
                 static_dim   = 127,
                 d_model      = 128,
                 nhead        = 4,
                 n_layers     = 2,
                 static_hidden= 64,
                 fusion_hidden= 128,
                 dropout      = 0.2,
                 max_seq_len  = 25):
        super().__init__()
        self.d_model = d_model

        self.vital_proj = nn.Linear(vital_dim, d_model)
        self.cls_token  = nn.Parameter(torch.zeros(1, 1, d_model))
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        self.pos_emb    = nn.Embedding(max_seq_len + 1, d_model)

        enc_layer = nn.TransformerEncoderLayer(
            d_model        = d_model,
            nhead          = nhead,
            dim_feedforward= d_model * 4,
            dropout        = dropout,
            batch_first    = True,
            norm_first     = True,
        )
        self.transformer = nn.TransformerEncoder(enc_layer, n_layers)

        self.static_enc = nn.Sequential(
            nn.Linear(static_dim, static_hidden),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.fusion = nn.Sequential(
            nn.LayerNorm(d_model + static_hidden),
            nn.Linear(d_model + static_hidden, fusion_hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(fusion_hidden, 1),  # single output
        )

    def forward(self, x_seq, x_static, lengths):
        B, T, _ = x_seq.shape

        x   = self.vital_proj(x_seq)
        cls = self.cls_token.expand(B, -1, -1)
        x   = torch.cat([cls, x], dim=1)
        pos = torch.arange(T + 1, device=x.device).unsqueeze(0)
        x   = x + self.pos_emb(pos)

        mask = torch.ones(
            B, T + 1, dtype=torch.bool, device=x.device
        )
        mask[:, 0] = False
        for i, l in enumerate(lengths):
            mask[i, 1:l.item() + 1] = False

        out     = self.transformer(x, src_key_padding_mask=mask)
        cls_out = out[:, 0, :]
        s_out   = self.static_enc(x_static)
        fused   = torch.cat([cls_out, s_out], dim=1)
        return torch.sigmoid(self.fusion(fused)).squeeze(1)


# ── Build datasets ────────────────────────────────────────────
print("Building datasets...")

for split in ['train', 'val', 'test']:
    sub = label_df[label_df['split'] == split]
    print(f"\n  {split}:")
    ds = RollingWindowDataset(
        sub, X_rich_full, X_static_all, stay_to_idx
    )
    if split == 'train':
        train_ds = ds
    elif split == 'val':
        val_ds   = ds
    else:
        test_ds  = ds

BATCH_SIZE = 512
train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE,
    shuffle=True, collate_fn=collate_rolling,
    num_workers=0, pin_memory=True
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE,
    shuffle=False, collate_fn=collate_rolling,
    num_workers=0, pin_memory=True
)
test_loader = DataLoader(
    test_ds, batch_size=BATCH_SIZE,
    shuffle=False, collate_fn=collate_rolling,
    num_workers=0, pin_memory=True
)

print(f"\nTrain batches : {len(train_loader):,}")
print(f"Val batches   : {len(val_loader):,}")
print(f"Test batches  : {len(test_loader):,}")

# ── Instantiate model ─────────────────────────────────────────
model = RollingTransformer(
    vital_dim    = len(rich_feature_names),
    static_dim   = len(feature_cols),
    d_model      = 128,
    nhead        = 4,
    n_layers     = 2,
    static_hidden= 64,
    fusion_hidden= 128,
    dropout      = 0.2,
    max_seq_len  = 25,
).to(device)

n_params = sum(p.numel() for p in model.parameters()
               if p.requires_grad)
print(f"\nModel parameters : {n_params:,}")
print("Architecture ready ✓")

Building datasets...

  train:
  Dataset built : 1,596,043 samples (0 skipped)

  val:
  Dataset built : 349,718 samples (0 skipped)

  test:
  Dataset built : 350,688 samples (0 skipped)

Train batches : 3,118
Val batches   : 684
Test batches  : 685

Model parameters : 436,737
Architecture ready ✓


In [5]:
# ============================================================
# Cell 3: Focal loss + training functions
# Focal loss down-weights easy negatives, focuses on
# hard positives — better than weighted BCE for 1.5% imbalance
# ============================================================

class FocalLoss(nn.Module):
    """
    Focal loss for binary classification.
    FL(p) = -alpha * (1-p)^gamma * log(p)

    gamma=2 focuses training on hard examples.
    alpha handles class imbalance.
    """
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, preds, targets):
        bce   = nn.functional.binary_cross_entropy(
            preds, targets, reduction='none'
        )
        pt    = torch.where(targets == 1, preds, 1 - preds)
        alpha = torch.where(
            targets == 1,
            torch.tensor(self.alpha, device=preds.device),
            torch.tensor(1 - self.alpha, device=preds.device)
        )
        focal = alpha * (1 - pt) ** self.gamma * bce
        return focal.mean()


# ── Also define weighted BCE as fallback ──────────────────────
pos_weight_tensor = torch.tensor([pos_weight]).to(device)
bce_weighted = nn.BCELoss(
    weight=None  # we apply manually per batch
)

def weighted_bce(preds, targets, pos_w=pos_weight):
    weights = torch.where(
        targets == 1,
        torch.tensor(pos_w, device=preds.device),
        torch.tensor(1.0,   device=preds.device)
    )
    bce = nn.functional.binary_cross_entropy(
        preds, targets, weight=weights, reduction='mean'
    )
    return bce


# ── Combined loss: focal + weighted BCE ──────────────────────
focal_loss = FocalLoss(alpha=0.25, gamma=2.0)

def combined_loss(preds, targets, fl_weight=0.5):
    fl  = focal_loss(preds, targets)
    wbce= weighted_bce(preds, targets)
    return fl_weight * fl + (1 - fl_weight) * wbce


# ── Train one epoch ───────────────────────────────────────────
def train_epoch(model, loader, optimizer, device):
    model.train()
    total_loss = 0
    total_n    = 0

    for x_seq, x_static, lengths, labels in loader:
        x_seq    = x_seq.to(device)
        x_static = x_static.to(device)
        lengths  = lengths.to(device)
        labels   = labels.to(device)

        optimizer.zero_grad()
        preds = model(x_seq, x_static, lengths)
        loss  = combined_loss(preds, labels)

        if torch.isnan(loss):
            continue

        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item() * len(labels)
        total_n    += len(labels)

    return total_loss / max(total_n, 1)


# ── Validate one epoch ────────────────────────────────────────
def val_epoch(model, loader, device):
    model.eval()
    total_loss = 0
    total_n    = 0
    all_preds  = []
    all_labels = []

    with torch.no_grad():
        for x_seq, x_static, lengths, labels in loader:
            x_seq    = x_seq.to(device)
            x_static = x_static.to(device)
            lengths  = lengths.to(device)
            labels   = labels.to(device)

            preds = model(x_seq, x_static, lengths)
            loss  = combined_loss(preds, labels)

            if not torch.isnan(loss):
                total_loss += loss.item() * len(labels)
                total_n    += len(labels)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    from sklearn.metrics import roc_auc_score
    all_preds  = np.array(all_preds)
    all_labels = np.array(all_labels)

    try:
        auroc = roc_auc_score(all_labels, all_preds)
    except Exception:
        auroc = float('nan')

    return total_loss / max(total_n, 1), auroc


print("Loss functions defined ✓")
print(f"  Focal loss  : alpha=0.25, gamma=2.0")
print(f"  Weighted BCE: pos_weight={pos_weight:.1f}x")
print(f"  Combined    : 50% focal + 50% weighted BCE")

Loss functions defined ✓
  Focal loss  : alpha=0.25, gamma=2.0
  Weighted BCE: pos_weight=61.5x
  Combined    : 50% focal + 50% weighted BCE


In [6]:
# ============================================================
# Cell 4: Train RollingTransformer
# Expected time: ~30-60 min on GPU for 50 epochs
# ============================================================
import time

print("Training RollingTransformer...")
print("=" * 55)

optimizer = torch.optim.Adam(
    model.parameters(), lr=1e-4, weight_decay=1e-5
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max',   # maximise val AUROC
    factor=0.5, patience=5
)

EPOCHS       = 50
best_auroc   = 0.0
best_epoch   = 0
patience     = 10
patience_ctr = 0
history      = []

print(f"\n{'Epoch':>5} | {'Train Loss':>10} | "
      f"{'Val Loss':>9} | {'Val AUROC':>9} | {'LR':>8}")
print("-" * 55)

for epoch in range(1, EPOCHS + 1):
    t0         = time.time()
    train_loss = train_epoch(model, train_loader, optimizer, device)
    val_loss, val_auroc = val_epoch(model, val_loader, device)
    lr_now     = optimizer.param_groups[0]['lr']
    elapsed    = time.time() - t0

    history.append({
        'epoch'     : epoch,
        'train_loss': train_loss,
        'val_loss'  : val_loss,
        'val_auroc' : val_auroc,
    })

    print(f"{epoch:>5} | {train_loss:>10.4f} | "
          f"{val_loss:>9.4f} | {val_auroc:>9.4f} | "
          f"{lr_now:>8.6f}  ({elapsed:.0f}s)")

    scheduler.step(val_auroc)

    if val_auroc > best_auroc:
        best_auroc   = val_auroc
        best_epoch   = epoch
        patience_ctr = 0
        torch.save(
            model.state_dict(),
            str(OUTPUT_DIR / 'rolling_transformer_best.pt')
        )
    else:
        patience_ctr += 1
        if patience_ctr >= patience:
            print(f"\nEarly stopping at epoch {epoch} "
                  f"(best={best_epoch}, "
                  f"val_auroc={best_auroc:.4f})")
            break

print(f"\nBest epoch   : {best_epoch}")
print(f"Best val AUROC: {best_auroc:.4f}")

history_df = pd.DataFrame(history)
history_df.to_csv(
    str(OUTPUT_DIR / 'rolling_transformer_history.csv'),
    index=False
)
print("Saved → rolling_transformer_history.csv ✓")
print("Saved → rolling_transformer_best.pt ✓")

Training RollingTransformer...

Epoch | Train Loss |  Val Loss | Val AUROC |       LR
-------------------------------------------------------
    1 |     0.6698 |    0.6369 |    0.7205 | 0.000100  (224s)
    2 |     0.6468 |    0.6264 |    0.7338 | 0.000100  (230s)
    3 |     0.6389 |    0.6609 |    0.7295 | 0.000100  (224s)
    4 |     0.6327 |    0.6244 |    0.7439 | 0.000100  (227s)
    5 |     0.6285 |    0.6134 |    0.7522 | 0.000100  (225s)
    6 |     0.6246 |    0.6110 |    0.7497 | 0.000100  (227s)
    7 |     0.6204 |    0.6088 |    0.7576 | 0.000100  (225s)
    8 |     0.6172 |    0.6106 |    0.7561 | 0.000100  (227s)
    9 |     0.6142 |    0.6103 |    0.7585 | 0.000100  (227s)
   10 |     0.6107 |    0.6111 |    0.7638 | 0.000100  (224s)
   11 |     0.6072 |    0.6099 |    0.7612 | 0.000100  (224s)
   12 |     0.6029 |    0.6144 |    0.7578 | 0.000100  (226s)
   13 |     0.6008 |    0.6679 |    0.7582 | 0.000100  (227s)
   14 |     0.5978 |    0.5968 |    0.7669 | 0.00010

In [7]:
# ============================================================
# Cell 5: Load best model and evaluate on test set
# Full rolling evaluation with alert tier analysis
# ============================================================
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    roc_curve, precision_recall_curve,
    confusion_matrix, f1_score
)

print("Loading best model...")
model.load_state_dict(torch.load(
    str(OUTPUT_DIR / 'rolling_transformer_best.pt'),
    map_location=device,
    weights_only=True,
))
model.eval()

# ── Collect all test predictions ─────────────────────────────
print("Evaluating on test set...")
all_preds  = []
all_labels = []

with torch.no_grad():
    for x_seq, x_static, lengths, labels in test_loader:
        x_seq    = x_seq.to(device)
        x_static = x_static.to(device)
        lengths  = lengths.to(device)
        preds    = model(x_seq, x_static, lengths)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)

# ── Overall metrics ───────────────────────────────────────────
auroc = roc_auc_score(all_labels, all_preds)
auprc = average_precision_score(all_labels, all_preds)

print(f"\n── RollingTransformer Test Results ──────────────────")
print(f"  AUROC      : {auroc:.4f}")
print(f"  AUPRC      : {auprc:.4f}")
print(f"  Test size  : {len(all_labels):,} stay-hours")
print(f"  Positives  : {int(all_labels.sum()):,} "
      f"({all_labels.mean():.3%})")

# ── Threshold analysis ────────────────────────────────────────
# Find Youden-optimal threshold
fpr, tpr, threshs = roc_curve(all_labels, all_preds)
youden_idx  = np.argmax(tpr - fpr)
best_thresh = threshs[youden_idx]
best_sens   = tpr[youden_idx]
best_spec   = 1 - fpr[youden_idx]

print(f"\n── Youden-optimal threshold ─────────────────────────")
print(f"  Threshold  : {best_thresh:.4f}")
print(f"  Sensitivity: {best_sens:.4f}")
print(f"  Specificity: {best_spec:.4f}")

y_pred = (all_preds >= best_thresh).astype(int)
tn, fp, fn, tp = confusion_matrix(all_labels, y_pred).ravel()
ppv = tp / (tp + fp + 1e-9)
npv = tn / (tn + fn + 1e-9)
f1  = f1_score(all_labels, y_pred)
print(f"  PPV        : {ppv:.4f}")
print(f"  NPV        : {npv:.4f}")
print(f"  F1         : {f1:.4f}")
print(f"  Alert rate : {y_pred.mean():.3%}")

# ── AUROC by hour range ───────────────────────────────────────
print(f"\n── AUROC by hour range ──────────────────────────────")
test_sub = label_df[label_df['split'] == 'test'].copy()
test_sub['pred'] = all_preds

print(f"{'Hour range':>15} {'Stay-hours':>12} "
      f"{'Positives':>10} {'AUROC':>8}")
print("-" * 50)

for lo, hi in [(4,12),(12,24),(24,48),(48,96),(96,200)]:
    sub   = test_sub[
        (test_sub['hour'] >= lo) &
        (test_sub['hour'] <  hi)
    ]
    n_pos = sub['label_12h'].sum()
    if n_pos < 10:
        print(f"  {lo:>3}h–{hi:>3}h : insufficient events")
        continue
    a = roc_auc_score(sub['label_12h'], sub['pred'])
    print(f"  {lo:>3}h–{hi:>3}h : "
          f"{len(sub):>10,}  {n_pos:>10,}  {a:>8.4f}")

# ── Save test predictions ─────────────────────────────────────
test_sub.to_csv(
    str(OUTPUT_DIR / 'rolling_test_predictions.csv'),
    index=False
)
print(f"\nSaved → rolling_test_predictions.csv ✓")

# ── Save final results ────────────────────────────────────────
import json
rolling_results = {
    'model'       : 'RollingTransformer',
    'auroc'       : round(float(auroc), 4),
    'auprc'       : round(float(auprc), 4),
    'best_thresh' : round(float(best_thresh), 4),
    'sensitivity' : round(float(best_sens), 4),
    'specificity' : round(float(best_spec), 4),
    'ppv'         : round(float(ppv), 4),
    'npv'         : round(float(npv), 4),
    'f1'          : round(float(f1), 4),
    'alert_rate'  : round(float(y_pred.mean()), 4),
    'n_test'      : int(len(all_labels)),
    'n_pos'       : int(all_labels.sum()),
}
with open(str(OUTPUT_DIR / 'rolling_results.json'), 'w') as f:
    json.dump(rolling_results, f, indent=2)
print("Saved → rolling_results.json ✓")
print("\n03e complete ✓")
print("Ready for updated 04 — rolling evaluation figures")

Loading best model...
Evaluating on test set...

── RollingTransformer Test Results ──────────────────
  AUROC      : 0.7558
  AUPRC      : 0.0771
  Test size  : 350,688 stay-hours
  Positives  : 5,248 (1.496%)

── Youden-optimal threshold ─────────────────────────
  Threshold  : 0.3856
  Sensitivity: 0.6446
  Specificity: 0.7459
  PPV        : 0.0371
  NPV        : 0.9928
  F1         : 0.0702
  Alert rate : 25.996%

── AUROC by hour range ──────────────────────────────
     Hour range   Stay-hours  Positives    AUROC
--------------------------------------------------
    4h– 12h :     64,178       2,220    0.7778
   12h– 24h :     76,677         890    0.7159
   24h– 48h :     91,875         893    0.6835
   48h– 96h :     75,437         755    0.6808
   96h–200h :     42,521         490    0.7062

Saved → rolling_test_predictions.csv ✓
Saved → rolling_results.json ✓

03e complete ✓
Ready for updated 04 — rolling evaluation figures
